## ESM scoring

In [21]:
reference_protein = "KTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQR"
protein_position = 200  # 1-based

index = protein_position - 1

masked_protein = (
    reference_protein[:index]
    + "_"
    + reference_protein[index + 1:]
)

print(masked_protein)
# MKTA_IAKQR

from proto_tools import (
    ESM2SampleConfig,
    ESM2SampleInput,
    run_esm2_sample,
)

inputs = ESM2SampleInput(
    sequences=[masked_protein]
)

config = ESM2SampleConfig(
    model_checkpoint="esm2_t30_150M_UR50D",
    batch_size=1,
    sampling_method="single_pass",
    return_logits=True,
    device="cpu",
)

result = run_esm2_sample(
    inputs,
    config,
)

print("Masked input:", masked_protein)
print("Sampled sequence:", result.sequences[0])

Starting worker
Running esm2


KTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRM_TAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQR
Masked input: KTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRM_TAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQR
Sampled sequence: KTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAYIAKQRMKTAY

In [22]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools

from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    run_esm2_score,
)

# Display docs
display_api_reference("esm2", "input", "run_esm2_score")

# Two short sequences to score
inputs = ESM2ScoringInput(sequences=["MKTAYIAKQR", "EVQLVESGGS"])

# Display config docs
display_api_reference("esm2", "config", "run_esm2_score")

# Process 5 masked variants per forward pass | see docs above for all fields
config = ESM2ScoringConfig(
    batch_size=5,
    model_checkpoint='esm2_t30_150M_UR50D',
    return_logits=False,
    device="cpu",  # Change to "cpu" if no GPU available
)

# Run the scoring function
score_result = run_esm2_score(inputs, config)

# Display output docs
display_api_reference("esm2", "output", "run_esm2_score")

# Show all scoring metrics for each input sequence
for i, score in enumerate(score_result.scores):
    print(f"Sequence {i + 1}: {inputs.sequences[i]}")
    print(f"  Log-likelihood:     {score['log_likelihood']:.3f}")
    print(f"  Avg log-likelihood: {score['avg_log_likelihood']:.3f}")
    print(f"  Perplexity:         {score['perplexity']:.3f}")

**Input** — `ESM2ScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Protein sequence(s) to process as string or list of strings. (will be normalized to List[str]) |

**Config** — `ESM2ScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>model_checkpoint</code> | <code>Literal['esm2_t6_8M_UR50D', 'esm2_t12_35M_UR50D', 'esm2_t30_150M_UR50D', 'esm2_t33_650M_UR50D', 'esm2_t36_3B_UR50D', 'esm2_t48_15B_UR50D']</code> | <code>'esm2_t33_650M_UR50D'</code> | ESM2 weights variant; trade off speed vs scoring fidelity |

Running run_esm2_score [00:00]

**Output** — `MaskedModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[MaskedModelScoringMetrics]</code> | required | List of scoring outputs, one per input sequence |

**Metrics** — `MaskedModelScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  |  |

Sequence 1: MKTAYIAKQR
  Log-likelihood:     -27.175
  Avg log-likelihood: -2.718
  Perplexity:         15.143
Sequence 2: EVQLVESGGS
  Log-likelihood:     -28.811
  Avg log-likelihood: -2.881
  Perplexity:         17.833


In [25]:
import numpy as np
import pandas as pd

from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    ESM2SampleConfig,
    ESM2SampleInput,
    run_esm2_score,
    run_esm2_sample,
)

MODEL = "esm2_t30_150M_UR50D"
DEVICE = "cpu"

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"


# Each row is one separate mutation
variants = [
    ("Protein 1", "MKTAYIAKQR", 2, "R"),
    ("Protein 1", "MKTAYIAKQR", 5, "F"),
    ("Protein 1", "MKTAYIAKQR", 8, "E"),

    ("Protein 2", "EVQLVESGGS", 2, "I"),
    ("Protein 2", "EVQLVESGGS", 6, "D"),
    ("Protein 2", "EVQLVESGGS", 9, "A"),
]


def make_mutant(sequence, position, mutant_aa):
    return sequence[:position - 1] + mutant_aa + sequence[position:]


def log_softmax(logits):
    logits = np.array(logits)
    logits = logits - logits.max()
    return logits - np.log(np.exp(logits).sum())


# ------------------------------------------------------------
# Create each single-mutant sequence
# ------------------------------------------------------------

prepared = []

for protein, reference, position, mutant_aa in variants:
    reference_aa = reference[position - 1]
    mutant = make_mutant(reference, position, mutant_aa)
    masked = reference[:position - 1] + "_" + reference[position:]

    prepared.append({
        "protein": protein,
        "mutation": f"{reference_aa}{position}{mutant_aa}",
        "position": position,
        "reference": reference,
        "mutant": mutant,
        "masked": masked,
        "reference_aa": reference_aa,
        "mutant_aa": mutant_aa,
    })

    print(f"\n{protein} — {reference_aa}{position}{mutant_aa}")
    print("Reference:", reference)
    print("Mutant:   ", mutant)


# ------------------------------------------------------------
# OLD METHOD: score the entire reference and mutant proteins
# ------------------------------------------------------------

exact_sequences = []

for variant in prepared:
    exact_sequences.extend([
        variant["reference"],
        variant["mutant"],
    ])

exact_result = run_esm2_score(
    ESM2ScoringInput(sequences=exact_sequences),
    ESM2ScoringConfig(
        model_checkpoint=MODEL,
        batch_size=5,
        device=DEVICE,
    ),
)


# ------------------------------------------------------------
# FAST METHOD: score only the mutated position
# ------------------------------------------------------------

fast_result = run_esm2_sample(
    ESM2SampleInput(
        sequences=[variant["masked"] for variant in prepared]
    ),
    ESM2SampleConfig(
        model_checkpoint=MODEL,
        batch_size=6,
        sampling_method="single_pass",
        return_logits=True,
        device=DEVICE,
    ),
)


# ------------------------------------------------------------
# Build one result row per mutation
# ------------------------------------------------------------

rows = []

for i, variant in enumerate(prepared):

    # Whole-sequence scores
    exact_reference = exact_result.scores[i * 2]["avg_log_likelihood"]
    exact_mutant = exact_result.scores[i * 2 + 1]["avg_log_likelihood"]

    # Position-specific scores
    position_logits = fast_result.logits[i][variant["position"] - 1]
    log_probs = log_softmax(position_logits)

    fast_reference = log_probs[
        AA_ORDER.index(variant["reference_aa"])
    ]

    fast_mutant = log_probs[
        AA_ORDER.index(variant["mutant_aa"])
    ]

    rows.append({
        "protein": variant["protein"],
        "mutation": variant["mutation"],
        "reference_sequence": variant["reference"],
        "mutant_sequence": variant["mutant"],

        "exact_reference_score": exact_reference,
        "exact_mutant_score": exact_mutant,
        "exact_difference": exact_mutant - exact_reference,

        "fast_reference_score": fast_reference,
        "fast_mutant_score": fast_mutant,
        "fast_difference": fast_mutant - fast_reference,
    })


results = pd.DataFrame(rows)


# Rank 1 = mutation predicted to be most damaging
results["exact_rank"] = results["exact_difference"].rank(
    ascending=True
).astype(int)

results["fast_rank"] = results["fast_difference"].rank(
    ascending=True
).astype(int)


display(results.round(4))


# ------------------------------------------------------------
# Compare the final rankings
# ------------------------------------------------------------

exact_order = results.sort_values(
    "exact_rank"
)[["protein", "mutation"]]

fast_order = results.sort_values(
    "fast_rank"
)[["protein", "mutation"]]

print("\nExact whole-sequence ranking:")
display(exact_order)

print("\nFast position-specific ranking:")
display(fast_order)

same_ranking = (
    exact_order["mutation"].tolist()
    == fast_order["mutation"].tolist()
)

print("\nSame ranking:", same_ranking)


Protein 1 — K2R
Reference: MKTAYIAKQR
Mutant:    MRTAYIAKQR

Protein 1 — Y5F
Reference: MKTAYIAKQR
Mutant:    MKTAFIAKQR

Protein 1 — K8E
Reference: MKTAYIAKQR
Mutant:    MKTAYIAEQR

Protein 2 — V2I
Reference: EVQLVESGGS
Mutant:    EIQLVESGGS

Protein 2 — E6D
Reference: EVQLVESGGS
Mutant:    EVQLVDSGGS

Protein 2 — G9A
Reference: EVQLVESGGS
Mutant:    EVQLVESGAS


Running run_esm2_score [00:00]

Starting worker
Running esm2
Exception in thread Thread-65 (_drain_stderr):
Traceback (most recent call last):
  File "/Users/marcamil/anaconda3/envs/proto/lib/python3.11/threading.py", line 1045, in _bootstrap_inner


,protein,mutation,reference_sequence,mutant_sequence,exact_reference_score,exact_mutant_score,exact_difference,fast_reference_score,fast_mutant_score,fast_difference,exact_rank,fast_rank
0,Protein 1,K2R,MKTAYIAKQR,MRTAYIAKQR,-2.7175,-2.7265,-0.0089,-2.6558,-2.7976,-0.1418,4,5
1,Protein 1,Y5F,MKTAYIAKQR,MKTAFIAKQR,-2.7175,-2.7170,0.0006,-3.7705,-3.3823,0.3881,5,6
2,Protein 1,K8E,MKTAYIAKQR,MKTAYIAEQR,-2.7175,-2.7600,-0.0425,-2.7567,-3.1247,-0.3680,2,2
3,Protein 2,V2I,EVQLVESGGS,EIQLVESGGS,-2.8811,-2.9402,-0.0591,-2.5107,-2.9624,-0.4517,1,1
4,Protein 2,E6D,EVQLVESGGS,EVQLVDSGGS,-2.8811,-2.8696,0.0114,-2.9372,-3.1756,-0.2384,6,3
5,Protein 2,G9A,EVQLVESGGS,EVQLVESGAS,-2.8811,-2.9130,-0.0319,-2.5252,-2.7395,-0.2143,3,4



Exact whole-sequence ranking:


...


Fast position-specific ranking:


    self._target(*self._args, **self._kwargs)
  File "/Users/marcamil/anaconda3/envs/proto/lib/python3.11/site-packages/proto_tools/utils/persistent_worker.py", line 672, in _drain_stderr


,protein,mutation
3,Protein 2,V2I
2,Protein 1,K8E
4,Protein 2,E6D
5,Protein 2,G9A
0,Protein 1,K2R
1,Protein 1,Y5F



Same ranking: False


    _drain_subprocess_stderr(self._process.stderr, parent_logger, self._stderr_lines, raw_tee)
  File "/Users/marcamil/anaconda3/envs/proto/lib/python3.11/site-packages/proto_tools/utils/persistent_worker.py", line 187, in _drain_subprocess_stderr
    for line in stream:
ValueError: I/O operation on closed file


In [27]:
import time
import numpy as np
import pandas as pd

from proto_tools import (
    ESM2ScoringConfig,
    ESM2ScoringInput,
    ESM2SampleConfig,
    ESM2SampleInput,
    run_esm2_score,
    run_esm2_sample,
)


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

MODEL = "esm2_t30_150M_UR50D"   # Small model for a quick CPU test
DEVICE = "cpu"
N_SAMPLED_POSITIONS = 10

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"


# ------------------------------------------------------------
# Create 30 test proteins of different lengths
# ------------------------------------------------------------

rng = np.random.default_rng(42)
lengths = range(40, 190, 5)   # 40, 45, ..., 185 = 30 proteins

proteins = pd.DataFrame({
    "protein": [f"Protein_{i + 1}" for i in range(30)],
    "sequence": [
        "".join(rng.choice(list(AA_ORDER), size=length))
        for length in lengths
    ],
})

proteins["length"] = proteins["sequence"].str.len()

print("Test proteins:")
display(proteins)


# ------------------------------------------------------------
# 1. Full scoring: score every position
# ------------------------------------------------------------

start = time.perf_counter()

full_result = run_esm2_score(
    ESM2ScoringInput(
        sequences=proteins["sequence"].tolist()
    ),
    ESM2ScoringConfig(
        model_checkpoint=MODEL,
        batch_size=32,
        device=DEVICE,
        return_logits=False,
    ),
)

full_time = time.perf_counter() - start

full_scores = [
    score["avg_log_likelihood"]
    for score in full_result.scores
]


# ------------------------------------------------------------
# 2. Sampled scoring: score only 16 positions
# ------------------------------------------------------------

masked_sequences = []
masked_info = []

for protein_index, sequence in enumerate(proteins["sequence"]):

    # Same evenly spaced relative positions for every protein
    positions = np.linspace(
        0,
        len(sequence) - 1,
        N_SAMPLED_POSITIONS,
        dtype=int,
    )

    for position in positions:
        masked_sequence = (
            sequence[:position]
            + "_"
            + sequence[position + 1:]
        )

        masked_sequences.append(masked_sequence)
        masked_info.append(
            (protein_index, position, sequence[position])
        )


start = time.perf_counter()

sampled_result = run_esm2_sample(
    ESM2SampleInput(sequences=masked_sequences),
    ESM2SampleConfig(
        model_checkpoint=MODEL,
        batch_size=32,
        sampling_method="single_pass",
        return_logits=True,
        device=DEVICE,
    ),
)

sampled_time = time.perf_counter() - start


# ------------------------------------------------------------
# Get log-probability at each sampled position
# ------------------------------------------------------------

scores_by_protein = [[] for _ in range(len(proteins))]

for result_index, (protein_index, position, true_aa) in enumerate(
    masked_info
):
    logits = np.array(
        sampled_result.logits[result_index][position],
        dtype=float,
    )

    # Convert logits to log probabilities
    logits = logits - logits.max()
    log_probs = logits - np.log(np.exp(logits).sum())

    true_aa_score = log_probs[AA_ORDER.index(true_aa)]
    scores_by_protein[protein_index].append(true_aa_score)


sampled_scores = [
    np.mean(position_scores)
    for position_scores in scores_by_protein
]


# ------------------------------------------------------------
# Compare scores and ranks
# ------------------------------------------------------------

results = proteins.copy()

results["full_score"] = full_scores
results["sampled_score"] = sampled_scores

# Higher average log-likelihood = better sequence
results["full_rank"] = results["full_score"].rank(
    ascending=False
).astype(int)

results["sampled_rank"] = results["sampled_score"].rank(
    ascending=False
).astype(int)

results["rank_difference"] = abs(
    results["full_rank"] - results["sampled_rank"]
)

results = results.sort_values("full_rank")

display(
    results[
        [
            "protein",
            "length",
            "full_score",
            "sampled_score",
            "full_rank",
            "sampled_rank",
            "rank_difference",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# Ranking comparison
# ------------------------------------------------------------

spearman = results["full_score"].corr(
    results["sampled_score"],
    method="spearman",
)

full_top_5 = set(
    results.nsmallest(5, "full_rank")["protein"]
)

sampled_top_5 = set(
    results.nsmallest(5, "sampled_rank")["protein"]
)

same_complete_ranking = (
    results.sort_values("full_rank")["protein"].tolist()
    ==
    results.sort_values("sampled_rank")["protein"].tolist()
)


print(f"Full scoring time:    {full_time:.2f} seconds")
print(f"Sampled scoring time: {sampled_time:.2f} seconds")

print(f"\nSpearman rank correlation: {spearman:.3f}")
print(f"Top-5 overlap: {len(full_top_5 & sampled_top_5)}/5")
print(f"Exactly the same full ranking: {same_complete_ranking}")

Test proteins:


,protein,sequence,length
0,Protein_1,CSQKKVCQFCMYRSRSMDTLMIEWSPKTMKLFCNVCVTGP,40
1,Protein_2,ESRICYKVQSSEILLAMERQWRIYKHWICLSELDQLHFNQWKETP,45
2,Protein_3,RCHSTKTTIVGFQPDTETASSSQLRGSNLMNADFDKQQLVNCSNPNNCNS,50
3,Protein_4,HPAHKYFGKYVAFTCVGWGKQDNMSYQKKKTHEHADCSRQLREWMWELQLKEIFH,55
4,Protein_5,QPPIYCHDHYIWLQLGSYGSGRSKRGCCKWDLRFRHTNMELVASLRQKHPDNCPPCSKSA,60
5,Protein_6,ELEHQDQDENSEGWYNKHPNGAEYKLLSCCGLQLKWENMLTGYHMMTKGAITAVMDHNHDQQDGF,65
6,Protein_7,QRRWSHDPWYFFADNMIHTLTYHFYPGPMWGYWEEMAGKYYEVCRTVFVDMLHASHQKIVCRRIGAWWFA,70
7,Protein_8,DCTVEREINRVAEGHYSHYLMKDWAEFGDAQYDGMWQSNKEVTARCRTDKDNWCIYHDLMQHYMGYWLANNDPTD,75
8,Protein_9,RDRKAYENEWRTKLPSTAQDGTGSLFHMSPWVSPGKDIWKDQYVGLMFGFYRTTYDVCLNIDETHHWDWWNEEGNELDMA,80
9,Protein_10,RCEETCKNMQTITHYMRVSVMALEVFFFPNWKGAAIWMVDCTDCKWQCTTTWFYHTNSPPPSRDTMHMYVTLSIQPFGCDKLEKP,85


Running run_esm2_score [00:00]

Starting worker
Running esm2


,protein,length,full_score,sampled_score,full_rank,sampled_rank,rank_difference
2,Protein_3,50,-2.9740,-3.0644,1,4,3
13,Protein_14,105,-3.0490,-3.2177,2,9,7
28,Protein_29,180,-3.0511,-3.2910,3,17,14
8,Protein_9,80,-3.0593,-3.2536,4,14,10
22,Protein_23,150,-3.0655,-3.3559,5,23,18
20,Protein_21,140,-3.0691,-3.4548,6,27,21
17,Protein_18,125,-3.0742,-3.2341,7,12,5
4,Protein_5,60,-3.0747,-3.2334,8,11,3
16,Protein_17,120,-3.0753,-2.6570,9,1,8
21,Protein_22,145,-3.0779,-3.2086,10,8,2


Full scoring time:    488.67 seconds
Sampled scoring time: 28.70 seconds

Spearman rank correlation: 0.376
Top-5 overlap: 1/5
Exactly the same full ranking: False


## ViennaRNA

In [17]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
from proto_tools import ViennaRNAInput, ViennaRNAConfig, run_viennarna
# Display input API reference
display_api_reference("viennarna", "input", "run_viennarna")

# Five RNA sequences spanning a range of structural behaviors
inputs = ViennaRNAInput(sequences=[
    "GCGCTTTTGCGC",    # GC-rich stem with UUUU loop (stable hairpin)
    "GGGGAAACCCC",     # G-rich stem with AAA loop
    "CGCGAAACGCG",     # Alternating CG stem with AAA loop
    "TTTTAAAAATTTTT",  # Poly-U/A stretch (expected to be unstructured)
    "GCTAGCTAGC",      # Mixed sequence with moderate folding potential
])

# Display config API reference
display_api_reference("viennarna", "config", "run_viennarna")

# Configure ViennaRNA folding at physiological temperature
config = ViennaRNAConfig(
    temperature=37.0,  # Physiological temperature in Celsius
)
result = run_viennarna(inputs, config)

# Display output API reference
display_api_reference("viennarna", "output", "run_viennarna")

# Display predicted structures and MFE for each sequence
for i, res in enumerate(result.results):
    print(f"Sequence {i + 1}: {res.sequence}")
    print(f"  Structure: {res.structure}")
    print(f"  MFE:       {res.mfe:.2f} kcal/mol")
    print()

**Input** — `ViennaRNAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | List of input RNA sequences |

**Config** — `ViennaRNAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>temperature</code> | <code>float</code> | <code>37.0</code> | Temperature in Celsius for energy calculations |
| <code>use_dna_params</code> | <code>bool</code> | <code>False</code> | Use DNA energy parameters instead of RNA parameters |
| <code>no_lonely_pairs</code> | <code>bool</code> | <code>False</code> | Disallow lonely base pairs (helices of length 1) |
| <code>dangles</code> | <code>Literal[0, 1, 2, 3]</code> | <code>2</code> | Dangling-end treatment: 0=ignore, 1=minimal, 2=multibranch, 3=accurate |
| <code>circ</code> | <code>bool</code> | <code>False</code> | Treat sequence as circular (plasmids, viroids, circRNAs) |
| <code>max_bp_span</code> | <code>int</code> | <code>-1</code> | Max base-pair span in nt; -1 = no limit, positive forbids long-range pairs |

Running run_viennarna [00:00]

**Output** — `ViennaRNAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[ViennaRNAResult]</code> | required | List of ViennaRNA results |

**`ViennaRNAResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | The input RNA sequence |
| <code>structure</code> | <code>str &#124; None</code> | <code>None</code> | Predicted RNA secondary structure in dot-bracket notation |
| <code>mfe</code> | <code>float &#124; None</code> | <code>None</code> | Minimum free energy in kcal/mol; more negative values indicate a more stable structure |

Sequence 1: GCGCUUUUGCGC
  Structure: ((((....))))
  MFE:       -5.70 kcal/mol

Sequence 2: GGGGAAACCCC
  Structure: ((((...))))
  MFE:       -4.50 kcal/mol

Sequence 3: CGCGAAACGCG
  Structure: ((((...))))
  MFE:       -2.80 kcal/mol

Sequence 4: UUUUAAAAAUUUUU
  Structure: ..............
  MFE:       0.00 kcal/mol

Sequence 5: GCUAGCUAGC
  Structure: (((....)))
  MFE:       -0.20 kcal/mol



## Alphagenome

In [18]:
# @title Install AlphaGenome

# @markdown Run this cell to install AlphaGenome.
from IPython.display import clear_output
! pip install alphagenome
clear_output()

from alphagenome import colab_utils
from alphagenome.data import gene_annotation, genome, transcript, track_data
from alphagenome.models import dna_client, variant_scorers
from alphagenome.visualization import plot_components
import pandas as pd
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Load the model.

import os
ALPHA_GENOME_API_KEY = os.environ["ALPHAGENOME_API_KEY"]

dna_model = dna_client.create(ALPHA_GENOME_API_KEY)

HG38_GTF_FEATHER = (
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'hg38/gencode.v46.annotation.gtf.gz.feather'
)
MM10_GTF_FEATHER = (
    'https://storage.googleapis.com/alphagenome/reference/gencode/'
    'mm10/gencode.vM23.annotation.gtf.gz.feather'
)

# Initialize an empty dictionary to serve as a variant effect prediction cache.
_prediction_cache = {}

_transcript_extractor_cache = {}

# @title Score variant { run: "auto" }
organism = 'human'  # @param ["human", "mouse"] {type:"string"}
organism_map = {
    'human': dna_client.Organism.HOMO_SAPIENS,
    'mouse': dna_client.Organism.MUS_MUSCULUS,
}
organism = organism_map[organism]

# @markdown Specify the variant:
variant_chromosome = 'chr22'  # @param { type:"string" }
variant_position = 36201698  # @param { type:"integer" }
variant_reference_bases = 'A'  # @param { type:"string" }
variant_alternate_bases = 'C'  # @param { type:"string" }

variant = genome.Variant(
    chromosome=variant_chromosome,
    position=variant_position,
    reference_bases=variant_reference_bases,
    alternate_bases=variant_alternate_bases,
)

# @markdown Specify length of sequence around variant to predict:
sequence_length = '1MB'  # @param ["2KB", "16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[
    f'SEQUENCE_LENGTH_{sequence_length}'
]

# The input interval is derived from the variant (centered on it).
interval = variant.reference_interval.resize(sequence_length)

# Select only the recommended RNA-seq variant scorer
rna_seq_scorer = variant_scorers.RECOMMENDED_VARIANT_SCORERS["RNA_SEQ"]

# Score the variant for RNA-seq effects only
variant_scores = dna_model.score_variant(
    interval=interval,
    variant=variant,
    variant_scorers=[rna_seq_scorer],
)

# Convert the results into a Pandas dataframe
df_rna = variant_scorers.tidy_scores(
    variant_scores,
    match_gene_strand=True,
)

# Extra safeguard: retain only RNA-seq assay rows
df_rna = df_rna[
    df_rna["Assay title"]
    .astype(str)
    .str.contains("RNA-seq", case=False, na=False)
].copy()

# Add absolute quantile score for ranking effect strength
df_rna["abs_quantile_score"] = df_rna["quantile_score"].abs()

# Sort strongest predicted effects first
df_rna = df_rna.sort_values(
    "abs_quantile_score",
    ascending=False,
)

# Display useful columns
columns_to_show = [
    "gene_id",
    "gene_name",
    "gene_type",
    "gene_strand",
    "Assay title",
    "ontology_curie",
    "biosample_name",
    "biosample_type",
    "gtex_tissue",
    "raw_score",
    "quantile_score",
    "abs_quantile_score",
]

df_rna[columns_to_show]



,gene_id,gene_name,gene_type,gene_strand,Assay title,ontology_curie,biosample_name,biosample_type,gtex_tissue,raw_score,quantile_score,abs_quantile_score
706,ENSG00000100336,APOL4,protein_coding,-,polyA plus RNA-seq,UBERON:0002046,thyroid gland,tissue,Thyroid,-1.646677e+00,-9.999999e-01,9.999999e-01
610,ENSG00000100336,APOL4,protein_coding,-,total RNA-seq,UBERON:0002170,upper lobe of right lung,tissue,,-1.442204e+00,-9.999999e-01,9.999999e-01
596,ENSG00000100336,APOL4,protein_coding,-,polyA plus RNA-seq,UBERON:0002080,heart right ventricle,tissue,,-1.966850e+00,-9.999999e-01,9.999999e-01
597,ENSG00000100336,APOL4,protein_coding,-,total RNA-seq,UBERON:0002080,heart right ventricle,tissue,,-1.789447e+00,-9.999999e-01,9.999999e-01
598,ENSG00000100336,APOL4,protein_coding,-,polyA plus RNA-seq,UBERON:0002084,heart left ventricle,tissue,,-1.929938e+00,-9.999999e-01,9.999999e-01
...,...,...,...,...,...,...,...,...,...,...,...,...
11169,ENSG00000279652,ENSG00000279652,lncRNA,+,total RNA-seq,CL:0000900,"naive thymus-derived CD8-positive, alpha-beta T cell",primary_cell,,0.000000e+00,-3.494449e-11,3.494449e-11
7864,ENSG00000234688,CACNG2-DT,lncRNA,+,total RNA-seq,CL:0002594,smooth muscle cell of the umbilical artery,primary_cell,,0.000000e+00,-3.494449e-11,3.494449e-11
1725,ENSG00000100348,TXN2,protein_coding,-,total RNA-seq,UBERON:0002190,subcutaneous adipose tissue,tissue,,-1.490116e-07,-3.494449e-11,3.494449e-11
6856,ENSG00000229971,ENSG00000229971,lncRNA,+,polyA plus RNA-seq,UBERON:0000945,stomach,tissue,,-1.430511e-06,-3.494449e-11,3.494449e-11


In [9]:
import requests

# ============================================================
# Variant information — same format used by AlphaGenome
# ============================================================
variant_chromosome = "chr22"
variant_position = 36201698
variant_reference_bases = "A"
variant_alternate_bases = "C"

# Number of bases to extract on EACH side of the variant
window_bp = 50


def extract_variant_dna_window(
    chromosome,
    position,
    reference_bases,
    alternate_bases,
    flank_size=50,
):
    """
    Extract a GRCh38 reference DNA sequence around a variant and
    create the corresponding alternate sequence.

    Coordinates:
        position is 1-based, matching AlphaGenome/VCF coordinates.

    For a single-base variant:
        total sequence length = flank_size * 2 + 1

    Example:
        flank_size=50 -> 101 bp sequence
    """

    reference_bases = reference_bases.upper()
    alternate_bases = alternate_bases.upper()

    # Ensembl uses chromosome names such as "22" rather than "chr22"
    chromosome_ensembl = chromosome.replace("chr", "", 1)

    # Handle mitochondrial chromosome naming
    if chromosome_ensembl == "M":
        chromosome_ensembl = "MT"

    # Include the full reference allele plus the requested flanks
    start = position - flank_size
    end = position + len(reference_bases) - 1 + flank_size

    if start < 1:
        raise ValueError("The requested window starts before position 1.")

    # Always request the forward/reference strand
    region = f"{chromosome_ensembl}:{start}..{end}:1"

    url = (
        "https://rest.ensembl.org/"
        f"sequence/region/homo_sapiens/{region}"
    )

    response = requests.get(
        url,
        params={"coord_system_version": "GRCh38"},
        headers={
            "Content-Type": "text/plain",
            "Accept": "text/plain",
        },
        timeout=30,
    )

    response.raise_for_status()

    reference_sequence = response.text.strip().upper()

    expected_length = flank_size * 2 + len(reference_bases)

    if len(reference_sequence) != expected_length:
        raise ValueError(
            f"Unexpected sequence length: {len(reference_sequence)}. "
            f"Expected: {expected_length}."
        )

    # Position of the variant inside the extracted sequence
    variant_index = flank_size

    fetched_reference = reference_sequence[
        variant_index:
        variant_index + len(reference_bases)
    ]

    # Confirm that the genome sequence agrees with the supplied REF allele
    if fetched_reference != reference_bases:
        raise ValueError(
            f"Reference mismatch at {chromosome}:{position}. "
            f"Expected {reference_bases}, but GRCh38 returned "
            f"{fetched_reference}."
        )

    # Replace the reference allele with the alternate allele
    alternate_sequence = (
        reference_sequence[:variant_index]
        + alternate_bases
        + reference_sequence[
            variant_index + len(reference_bases):
        ]
    )

    metadata = {
        "chromosome": chromosome,
        "position": position,
        "region_start": start,
        "region_end": end,
        "flank_size": flank_size,
        "reference_allele": reference_bases,
        "alternate_allele": alternate_bases,
        "reference_length": len(reference_sequence),
        "alternate_length": len(alternate_sequence),
    }

    return reference_sequence, alternate_sequence, metadata


# ============================================================
# Extract the sequences
# ============================================================
reference_dna, alternate_dna, metadata = extract_variant_dna_window(
    chromosome=variant_chromosome,
    position=variant_position,
    reference_bases=variant_reference_bases,
    alternate_bases=variant_alternate_bases,
    flank_size=window_bp,
)


# ============================================================
# Display results
# ============================================================
print("Region:")
print(
    f"{metadata['chromosome']}:"
    f"{metadata['region_start']}-"
    f"{metadata['region_end']}"
)

print("\nReference DNA:")
print(reference_dna)

print("\nAlternate DNA:")
print(alternate_dna)

print("\nSequence lengths:")
print("Reference:", len(reference_dna))
print("Alternate:", len(alternate_dna))

# Show the variant in context
context_size = 10
variant_index = window_bp

print("\nReference variant context:")
print(
    reference_dna[variant_index - context_size:variant_index]
    + "["
    + reference_dna[
        variant_index:
        variant_index + len(variant_reference_bases)
    ]
    + "]"
    + reference_dna[
        variant_index + len(variant_reference_bases):
        variant_index + len(variant_reference_bases) + context_size
    ]
)

print("\nAlternate variant context:")
print(
    alternate_dna[variant_index - context_size:variant_index]
    + "["
    + alternate_dna[
        variant_index:
        variant_index + len(variant_alternate_bases)
    ]
    + "]"
    + alternate_dna[
        variant_index + len(variant_alternate_bases):
        variant_index + len(variant_alternate_bases) + context_size
    ]
)

Region:
chr22:36201648-36201748

Reference DNA:
CCAGGAGAGAGCAGAGGAGCAGGAGGGCAGAGAGCTGGGGCCTCGGACTCACCCGACGCTTGTGATGAGCTGCACCCAGGATCCCATCCTCCTTGGTCATT

Alternate DNA:
CCAGGAGAGAGCAGAGGAGCAGGAGGGCAGAGAGCTGGGGCCTCGGACTCCCCCGACGCTTGTGATGAGCTGCACCCAGGATCCCATCCTCCTTGGTCATT

Sequence lengths:
Reference: 101
Alternate: 101

Reference variant context:
CCTCGGACTC[A]CCCGACGCTT

Alternate variant context:
CCTCGGACTC[C]CCCGACGCTT


In [13]:
# Install once if needed:
# !pip install requests biopython pandas -q

import hashlib
import textwrap
import time
from collections import defaultdict
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from Bio.Seq import Seq


# ============================================================
# INPUTS
# ============================================================

gene_name = "APOL4"

variant_chromosome = "chr22"
variant_position = 36201698
variant_reference_bases = "A"
variant_alternate_bases = "C"

# Print every full protein sequence
PRINT_FULL_SEQUENCES = True

# Save CSV and FASTA files
SAVE_FILES = True

OUTPUT_DIR = Path("APOL4_isoforms_output")


# ============================================================
# ENSEMBL SETTINGS
# ============================================================

ENSEMBL_SERVER = "https://rest.ensembl.org"
SPECIES = "homo_sapiens"

SESSION = requests.Session()

SESSION.headers.update({
    "User-Agent": "DOGMA-isoform-mapper/1.0",
})


# ============================================================
# ENSEMBL API HELPERS
# ============================================================

def request_ensembl(
    endpoint,
    params=None,
    accept="application/json",
    retries=5,
):
    """
    Send a GET request to Ensembl with basic retry handling.
    """

    url = ENSEMBL_SERVER + endpoint
    last_error = None

    for attempt in range(retries):

        try:

            response = SESSION.get(
                url,
                params=params,
                headers={
                    "Content-Type": accept,
                    "Accept": accept,
                },
                timeout=60,
            )

            # Ensembl rate limit
            if response.status_code == 429:

                retry_after = response.headers.get(
                    "Retry-After",
                    1,
                )

                time.sleep(float(retry_after))
                continue

            # Temporary server error
            if 500 <= response.status_code < 600:

                time.sleep(2 ** attempt)
                continue

            response.raise_for_status()

            return response

        except requests.RequestException as exc:

            last_error = exc

            if attempt == retries - 1:
                break

            time.sleep(2 ** attempt)

    raise RuntimeError(
        f"Ensembl request failed: {url}"
    ) from last_error


def get_json(endpoint, params=None):
    """
    Retrieve JSON from Ensembl.
    """

    response = request_ensembl(
        endpoint,
        params=params,
        accept="application/json",
    )

    return response.json()


def get_sequence(
    identifier,
    sequence_type,
):
    """
    Retrieve a sequence from Ensembl.

    sequence_type may be:
        genomic
        cdna
        cds
        protein
    """

    response = request_ensembl(
        f"/sequence/id/{identifier}",
        params={
            "type": sequence_type,
        },
        accept="text/plain",
    )

    return response.text.strip().upper()


# ============================================================
# GENERAL HELPERS
# ============================================================

def strip_version(identifier):
    """
    Convert:

        ENST00000683024.1

    into:

        ENST00000683024
    """

    if identifier is None:
        return None

    return str(identifier).split(
        ".",
        1,
    )[0]


def normalize_chromosome(chromosome):
    """
    Convert chr22 to 22 and chrM to MT.
    """

    chromosome = str(chromosome)

    if chromosome.lower().startswith("chr"):
        chromosome = chromosome[3:]

    if chromosome == "M":
        chromosome = "MT"

    return chromosome


def reverse_complement(sequence):
    """
    Reverse-complement a DNA sequence.
    """

    return str(
        Seq(
            sequence.upper()
        ).reverse_complement()
    )


def translate_cds(cds_sequence):
    """
    Translate an annotated CDS into amino acids.

    The ordinary terminal stop codon is removed.

    If a premature stop occurs, '*' is retained and the
    protein is truncated at that position.
    """

    cds_sequence = cds_sequence.upper()

    # Remove an incomplete final codon if present
    usable_length = (
        len(cds_sequence) // 3
    ) * 3

    cds_sequence = cds_sequence[
        :usable_length
    ]

    protein = str(
        Seq(cds_sequence).translate(
            to_stop=False
        )
    )

    # Remove normal terminal stop
    if protein.endswith("*"):
        protein = protein[:-1]

    # Retain and truncate at premature stop
    if "*" in protein:

        protein = protein[
            :protein.index("*") + 1
        ]

    return protein


def sequence_hash(sequence):
    """
    Hash a protein sequence so identical sequences can be grouped.
    """

    return hashlib.sha256(
        sequence.encode()
    ).hexdigest()


def wrap_sequence(
    sequence,
    width=80,
):
    """
    Wrap a sequence for readable printing or FASTA output.
    """

    return "\n".join(
        textwrap.wrap(
            sequence,
            width=width,
        )
    )


def list_to_text(value):
    """
    Convert a list into comma-separated text for the CSV.
    """

    if not value:
        return ""

    if isinstance(value, list):

        return ",".join(
            str(item)
            for item in value
        )

    return str(value)


# ============================================================
# LOOK UP GENE AND ALL TRANSLATED TRANSCRIPTS
# ============================================================

def lookup_gene(symbol):
    """
    Look up a gene and expand all transcript records.
    """

    endpoint = (
        f"/lookup/symbol/"
        f"{SPECIES}/"
        f"{quote(symbol)}"
    )

    return get_json(
        endpoint,
        params={
            "expand": 1,
        },
    )


def collect_all_translated_transcripts(
    gene_record,
):
    """
    Keep every Ensembl transcript that contains a Translation.

    Multiple transcripts may encode the same exact protein.
    Those transcripts are grouped into unique protein isoforms
    later in the workflow.
    """

    canonical_id = strip_version(
        gene_record.get(
            "canonical_transcript"
        )
    )

    records = []

    for transcript in gene_record.get(
        "Transcript",
        [],
    ):

        translation = transcript.get(
            "Translation"
        )

        # Ignore transcripts with no translated protein
        if not translation:
            continue

        transcript_id = strip_version(
            transcript.get("id")
        )

        protein_id = strip_version(
            translation.get("id")
        )

        if not transcript_id or not protein_id:
            continue

        records.append({
            "transcript_id":
                transcript_id,

            "transcript_version":
                transcript.get("version"),

            "protein_id":
                protein_id,

            "protein_version":
                translation.get("version"),

            "transcript_biotype":
                transcript.get("biotype"),

            "strand_number":
                int(
                    transcript.get(
                        "strand",
                        gene_record["strand"],
                    )
                ),

            "is_ensembl_canonical":
                transcript_id
                == canonical_id,

            "annotated_translation_length":
                translation.get("length"),
        })

    if not records:

        raise ValueError(
            "No translated transcripts were "
            f"found for "
            f"{gene_record.get('display_name')}."
        )

    return records


# ============================================================
# ANNOTATE VARIANT ACROSS ALL TRANSCRIPTS
# ============================================================

def annotate_variant(
    chromosome,
    position,
    reference,
    alternate,
):
    """
    Annotate an equal-length substitution using Ensembl VEP.

    Examples supported:

        A>C
        AT>GC

    Insertions and deletions require additional coordinate logic.
    """

    reference = reference.upper()
    alternate = alternate.upper()

    if len(reference) != len(alternate):

        raise ValueError(
            "This version supports substitutions only. "
            "REF and ALT must have equal lengths."
        )

    chromosome = normalize_chromosome(
        chromosome
    )

    variant_end = (
        position
        + len(reference)
        - 1
    )

    region = (
        f"{chromosome}:"
        f"{position}-"
        f"{variant_end}"
    )

    endpoint = (
        f"/vep/{SPECIES}/region/"
        f"{region}/"
        f"{alternate}"
    )

    results = get_json(
        endpoint,
        params={
            "symbol": 1,
            "canonical": 1,
            "mane": 1,
            "appris": 1,
            "ccds": 1,
            "protein": 1,
            "hgvs": 1,
            "numbers": 1,
        },
    )

    if not results:

        raise ValueError(
            "Ensembl VEP returned no result."
        )

    result = results[0]

    # Confirm that REF matches GRCh38
    allele_string = result.get(
        "allele_string",
        "",
    )

    if "/" in allele_string:

        genome_reference = (
            allele_string
            .split("/", 1)[0]
            .upper()
        )

        if genome_reference != reference:

            raise ValueError(
                "GRCh38 reference mismatch:\n"
                f"Supplied REF: {reference}\n"
                f"Ensembl REF: {genome_reference}"
            )

    return result


def index_gene_consequences(
    vep_result,
    gene_id,
):
    """
    Create:

        transcript_id -> VEP consequence

    for the requested gene.
    """

    gene_id = strip_version(
        gene_id
    )

    consequence_index = {}

    for consequence in vep_result.get(
        "transcript_consequences",
        [],
    ):

        consequence_gene_id = strip_version(
            consequence.get("gene_id")
        )

        if consequence_gene_id != gene_id:
            continue

        transcript_id = strip_version(
            consequence.get(
                "transcript_id"
            )
        )

        if transcript_id:

            consequence_index[
                transcript_id
            ] = consequence

    return consequence_index


# ============================================================
# ANALYSE ONE TRANSCRIPT/ISOFORM
# ============================================================

def analyse_one_transcript(
    transcript,
    consequence,
    reference,
    alternate,
):
    """
    Retrieve one transcript-derived protein.

    If the genomic variant directly overlaps the transcript's CDS,
    create the alternate CDS and translate the alternate protein.

    For intronic, regulatory, or splice-only variants, the code does
    not invent a full alternate protein because the mature splicing
    outcome is unknown.
    """

    transcript_id = transcript[
        "transcript_id"
    ]

    protein_id = transcript[
        "protein_id"
    ]

    strand_number = transcript[
        "strand_number"
    ]

    reference_protein = get_sequence(
        protein_id,
        "protein",
    )

    result = {
        **transcript,

        "reference_protein":
            reference_protein,

        "reference_aa_length":
            len(reference_protein),

        "reference_hash":
            sequence_hash(
                reference_protein
            ),

        "mane_select":
            False,

        "mane_plus_clinical":
            False,

        "appris":
            None,

        "ccds":
            None,

        "consequence_terms":
            [],

        "direct_cds_overlap":
            False,

        "cds_position":
            None,

        "protein_position":
            None,

        "codon_change":
            None,

        "amino_acid_change":
            None,

        "reference_amino_acid":
            None,

        "alternate_amino_acid":
            None,

        "hgvsc":
            None,

        "hgvsp":
            None,

        "protein_changed":
            None,

        "alternate_protein":
            None,

        "alternate_aa_length":
            None,

        "alternate_hash":
            None,

        "alternate_status":
            "not_resolved_no_direct_cds_change",

        "mapping_error":
            None,
    }

    # The variant may not be associated with this transcript
    if consequence is None:

        result[
            "alternate_status"
        ] = (
            "no_vep_consequence_for_transcript"
        )

        return result

    consequence_terms = consequence.get(
        "consequence_terms",
        [],
    )

    result.update({
        "mane_select":
            bool(
                consequence.get(
                    "mane_select"
                )
            ),

        "mane_plus_clinical":
            bool(
                consequence.get(
                    "mane_plus_clinical"
                )
            ),

        "appris":
            consequence.get("appris"),

        "ccds":
            consequence.get("ccds"),

        "consequence_terms":
            consequence_terms,

        "codon_change":
            consequence.get("codons"),

        "amino_acid_change":
            consequence.get(
                "amino_acids"
            ),

        "hgvsc":
            consequence.get("hgvsc"),

        "hgvsp":
            consequence.get("hgvsp"),
    })

    cds_start = consequence.get(
        "cds_start"
    )

    cds_end = consequence.get(
        "cds_end"
    )

    protein_start = consequence.get(
        "protein_start"
    )

    # No CDS coordinate means no codon can be replaced directly
    if cds_start is None:
        return result

    result[
        "direct_cds_overlap"
    ] = True

    try:

        reference_cds = get_sequence(
            transcript_id,
            "cds",
        )

        # Ensembl CDS is already:
        #
        #   1. spliced
        #   2. oriented 5' -> 3'
        #
        # Therefore genomic alleles must be reverse-complemented
        # when the transcript is on the negative strand.

        if strand_number == -1:

            transcript_reference = (
                reverse_complement(
                    reference
                )
            )

            transcript_alternate = (
                reverse_complement(
                    alternate
                )
            )

        else:

            transcript_reference = (
                reference.upper()
            )

            transcript_alternate = (
                alternate.upper()
            )

        # VEP CDS positions are 1-based
        cds_start = int(cds_start)

        cds_end = int(
            cds_end
            if cds_end is not None
            else cds_start
        )

        cds_index = cds_start - 1

        observed_reference = (
            reference_cds[
                cds_index:
                cds_index
                + len(
                    transcript_reference
                )
            ]
        )

        if (
            observed_reference
            != transcript_reference
        ):

            raise ValueError(
                "CDS reference mismatch: "
                f"expected "
                f"{transcript_reference}, "
                f"observed "
                f"{observed_reference}"
            )

        # Create alternate CDS
        alternate_cds = (
            reference_cds[
                :cds_index
            ]
            + transcript_alternate
            + reference_cds[
                cds_index
                + len(
                    transcript_reference
                ):
            ]
        )

        # Translate alternate CDS
        alternate_protein = (
            translate_cds(
                alternate_cds
            )
        )

        protein_changed = (
            alternate_protein
            != reference_protein
        )

        result.update({
            "cds_position":
                (
                    str(cds_start)
                    if cds_start == cds_end
                    else (
                        f"{cds_start}-"
                        f"{cds_end}"
                    )
                ),

            "alternate_protein":
                alternate_protein,

            "alternate_aa_length":
                len(
                    alternate_protein
                ),

            "alternate_hash":
                sequence_hash(
                    alternate_protein
                ),

            "protein_changed":
                protein_changed,

            "alternate_status":
                (
                    "resolved_changed"
                    if protein_changed
                    else (
                        "resolved_same_as_reference"
                    )
                ),
        })

        if protein_start is not None:

            protein_position = int(
                protein_start
            )

            protein_index = (
                protein_position - 1
            )

            result[
                "protein_position"
            ] = protein_position

            if (
                protein_index
                < len(reference_protein)
            ):

                result[
                    "reference_amino_acid"
                ] = reference_protein[
                    protein_index
                ]

            if (
                protein_index
                < len(alternate_protein)
            ):

                result[
                    "alternate_amino_acid"
                ] = alternate_protein[
                    protein_index
                ]

        # A stop-lost protein may extend into the 3' UTR.
        # The annotated CDS alone cannot reconstruct that
        # complete extension.
        if "stop_lost" in consequence_terms:

            result[
                "alternate_status"
            ] = (
                "partial_stop_lost_"
                "extension_not_reconstructed"
            )

    except Exception as exc:

        result[
            "mapping_error"
        ] = str(exc)

        result[
            "alternate_status"
        ] = "mapping_failed"

        result[
            "alternate_protein"
        ] = None

        result[
            "protein_changed"
        ] = None

    return result


# ============================================================
# GROUP TRANSCRIPTS INTO UNIQUE PROTEIN ISOFORMS
# ============================================================

def assign_unique_reference_isoforms(
    results,
):
    """
    Group transcripts by exact reference amino-acid sequence.

    Example:

        ENST_A -> same 348-aa sequence
        ENST_B -> same 348-aa sequence

    These are assigned to the same unique protein isoform.
    """

    groups = defaultdict(list)

    for result in results:

        groups[
            result["reference_hash"]
        ].append(result)

    ordered_groups = sorted(
        groups.values(),
        key=lambda group: (
            not any(
                item[
                    "is_ensembl_canonical"
                ]
                for item in group
            ),
            not any(
                item["mane_select"]
                for item in group
            ),
            -group[0][
                "reference_aa_length"
            ],
            group[0][
                "transcript_id"
            ],
        ),
    )

    for number, group in enumerate(
        ordered_groups,
        start=1,
    ):

        isoform_id = (
            f"ISOFORM_{number}"
        )

        for result in group:

            result[
                "unique_reference_isoform"
            ] = isoform_id

    return ordered_groups


def assign_unique_alternate_isoforms(
    results,
):
    """
    Group variant-resolved alternate proteins by exact sequence.
    """

    groups = defaultdict(list)

    for result in results:

        if (
            result[
                "alternate_protein"
            ]
            is not None
        ):

            groups[
                result["alternate_hash"]
            ].append(result)

    ordered_groups = sorted(
        groups.values(),
        key=lambda group: (
            -group[0][
                "alternate_aa_length"
            ],
            group[0][
                "transcript_id"
            ],
        ),
    )

    for number, group in enumerate(
        ordered_groups,
        start=1,
    ):

        isoform_id = (
            f"ALT_ISOFORM_{number}"
        )

        for result in group:

            result[
                "unique_alternate_isoform"
            ] = isoform_id

    for result in results:

        result.setdefault(
            "unique_alternate_isoform",
            None,
        )

    return ordered_groups


# ============================================================
# MAIN ANALYSIS
# ============================================================

def analyse_all_isoforms(
    gene_symbol,
    chromosome,
    position,
    reference,
    alternate,
):
    """
    Retrieve and analyse all translated isoforms for a gene.
    """

    gene_symbol = gene_symbol.upper()
    reference = reference.upper()
    alternate = alternate.upper()

    # Look up gene and all transcripts
    gene = lookup_gene(
        gene_symbol
    )

    gene_id = strip_version(
        gene["id"]
    )

    transcripts = (
        collect_all_translated_transcripts(
            gene
        )
    )

    # Annotate variant across all transcripts
    vep_result = annotate_variant(
        chromosome,
        position,
        reference,
        alternate,
    )

    consequence_index = (
        index_gene_consequences(
            vep_result,
            gene_id,
        )
    )

    results = []

    for transcript in transcripts:

        transcript_id = transcript[
            "transcript_id"
        ]

        consequence = (
            consequence_index.get(
                transcript_id
            )
        )

        transcript_result = (
            analyse_one_transcript(
                transcript=transcript,
                consequence=consequence,
                reference=reference,
                alternate=alternate,
            )
        )

        results.append(
            transcript_result
        )

    reference_groups = (
        assign_unique_reference_isoforms(
            results
        )
    )

    alternate_groups = (
        assign_unique_alternate_isoforms(
            results
        )
    )

    # Sort transcript table:
    # canonical first, then MANE, then isoform
    results.sort(
        key=lambda item: (
            not item[
                "is_ensembl_canonical"
            ],
            not item[
                "mane_select"
            ],
            item[
                "unique_reference_isoform"
            ],
            item[
                "transcript_id"
            ],
        )
    )

    gene_chromosome = str(
        gene["seq_region_name"]
    )

    variant_inside_gene = (
        normalize_chromosome(
            chromosome
        )
        == normalize_chromosome(
            gene_chromosome
        )
        and int(
            gene["start"]
        )
        <= position
        <= int(
            gene["end"]
        )
    )

    return {
        "gene_name":
            gene_symbol,

        "gene_id":
            gene_id,

        "gene_chromosome":
            gene_chromosome,

        "gene_start":
            int(gene["start"]),

        "gene_end":
            int(gene["end"]),

        "gene_strand":
            (
                "+"
                if int(gene["strand"]) == 1
                else "-"
            ),

        "variant":
            (
                f"{chromosome}:"
                f"{position}:"
                f"{reference}>"
                f"{alternate}"
            ),

        "variant_inside_gene_span":
            variant_inside_gene,

        "most_severe_consequence":
            vep_result.get(
                "most_severe_consequence"
            ),

        "results":
            results,

        "reference_groups":
            reference_groups,

        "alternate_groups":
            alternate_groups,
    }


# ============================================================
# CREATE SUMMARY DATAFRAME
# ============================================================

def make_summary_dataframe(
    results,
):
    """
    Create one row per translated transcript.
    """

    rows = []

    for result in results:

        rows.append({
            "unique_reference_isoform":
                result[
                    "unique_reference_isoform"
                ],

            "unique_alternate_isoform":
                result[
                    "unique_alternate_isoform"
                ],

            "transcript_id":
                result["transcript_id"],

            "transcript_version":
                result[
                    "transcript_version"
                ],

            "protein_id":
                result["protein_id"],

            "protein_version":
                result[
                    "protein_version"
                ],

            "transcript_biotype":
                result[
                    "transcript_biotype"
                ],

            "is_ensembl_canonical":
                result[
                    "is_ensembl_canonical"
                ],

            "mane_select":
                result["mane_select"],

            "mane_plus_clinical":
                result[
                    "mane_plus_clinical"
                ],

            "appris":
                result["appris"],

            "ccds":
                result["ccds"],

            "reference_aa_length":
                result[
                    "reference_aa_length"
                ],

            "consequence_terms":
                list_to_text(
                    result[
                        "consequence_terms"
                    ]
                ),

            "direct_cds_overlap":
                result[
                    "direct_cds_overlap"
                ],

            "cds_position":
                result[
                    "cds_position"
                ],

            "protein_position":
                result[
                    "protein_position"
                ],

            "codon_change":
                result[
                    "codon_change"
                ],

            "amino_acid_change":
                result[
                    "amino_acid_change"
                ],

            "reference_amino_acid":
                result[
                    "reference_amino_acid"
                ],

            "alternate_amino_acid":
                result[
                    "alternate_amino_acid"
                ],

            "hgvsc":
                result["hgvsc"],

            "hgvsp":
                result["hgvsp"],

            "protein_changed":
                result[
                    "protein_changed"
                ],

            "alternate_status":
                result[
                    "alternate_status"
                ],

            "mapping_error":
                result[
                    "mapping_error"
                ],

            "reference_protein":
                result[
                    "reference_protein"
                ],

            "alternate_protein":
                result[
                    "alternate_protein"
                ],
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# PRINT RESULTS
# ============================================================

def print_results(
    analysis,
    summary_df,
):
    print("=" * 80)
    print("GENE AND VARIANT")
    print("=" * 80)

    print(
        "Gene:",
        analysis["gene_name"],
    )

    print(
        "Ensembl gene ID:",
        analysis["gene_id"],
    )

    print(
        "Gene location:",
        f"chr{analysis['gene_chromosome']}:"
        f"{analysis['gene_start']}-"
        f"{analysis['gene_end']}",
    )

    print(
        "Gene strand:",
        analysis["gene_strand"],
    )

    print(
        "Variant:",
        analysis["variant"],
    )

    print(
        "Variant inside gene span:",
        analysis[
            "variant_inside_gene_span"
        ],
    )

    print(
        "Most severe VEP consequence:",
        analysis[
            "most_severe_consequence"
        ],
    )

    print(
        "Translated Ensembl transcripts:",
        len(
            analysis["results"]
        ),
    )

    print(
        "Unique reference protein isoforms:",
        len(
            analysis[
                "reference_groups"
            ]
        ),
    )

    print(
        "Variant-resolved alternate protein isoforms:",
        len(
            analysis[
                "alternate_groups"
            ]
        ),
    )

    display_columns = [
        "unique_reference_isoform",
        "transcript_id",
        "protein_id",
        "reference_aa_length",
        "is_ensembl_canonical",
        "mane_select",
        "appris",
        "ccds",
        "consequence_terms",
        "direct_cds_overlap",
        "amino_acid_change",
        "hgvsp",
        "protein_changed",
        "alternate_status",
    ]

    print("\nTRANSCRIPT-TO-ISOFORM SUMMARY")

    try:

        from IPython.display import display

        display(
            summary_df[
                display_columns
            ]
        )

    except ImportError:

        print(
            summary_df[
                display_columns
            ].to_string(
                index=False
            )
        )

    if not PRINT_FULL_SEQUENCES:
        return

    # --------------------------------------------------------
    # Print each unique reference protein once
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("UNIQUE REFERENCE PROTEIN ISOFORMS")
    print("=" * 80)

    for group in analysis[
        "reference_groups"
    ]:

        representative = group[0]

        sequence = representative[
            "reference_protein"
        ]

        transcripts = ", ".join(
            item["transcript_id"]
            for item in group
        )

        proteins = ", ".join(
            item["protein_id"]
            for item in group
        )

        print(
            f"\n>"
            f"{analysis['gene_name']}|"
            f"{representative['unique_reference_isoform']}"
        )

        print(
            "Transcripts:",
            transcripts,
        )

        print(
            "Protein IDs:",
            proteins,
        )

        print(
            "Length:",
            len(sequence),
            "aa",
        )

        print(
            "Ensembl canonical:",
            any(
                item[
                    "is_ensembl_canonical"
                ]
                for item in group
            ),
        )

        print(
            "MANE Select:",
            any(
                item[
                    "mane_select"
                ]
                for item in group
            ),
        )

        print(
            wrap_sequence(
                sequence
            )
        )

    # --------------------------------------------------------
    # Print directly resolvable alternate proteins
    # --------------------------------------------------------

    print("\n" + "=" * 80)
    print("VARIANT-RESOLVED ALTERNATE PROTEINS")
    print("=" * 80)

    if not analysis[
        "alternate_groups"
    ]:

        print(
            "No complete alternate protein could be "
            "directly resolved for any isoform."
        )

    for group in analysis[
        "alternate_groups"
    ]:

        representative = group[0]

        sequence = representative[
            "alternate_protein"
        ]

        transcripts = ", ".join(
            item["transcript_id"]
            for item in group
        )

        statuses = ", ".join(
            sorted({
                item[
                    "alternate_status"
                ]
                for item in group
            })
        )

        print(
            f"\n>"
            f"{analysis['gene_name']}|"
            f"{representative['unique_alternate_isoform']}|"
            f"{analysis['variant']}"
        )

        print(
            "Transcripts:",
            transcripts,
        )

        print(
            "Length:",
            len(sequence),
            "aa",
        )

        print(
            "Status:",
            statuses,
        )

        print(
            wrap_sequence(
                sequence
            )
        )


# ============================================================
# FASTA OUTPUT
# ============================================================

def write_fasta(
    groups,
    output_path,
    gene,
    sequence_key,
    isoform_key,
    variant=None,
):
    """
    Write unique protein sequences to FASTA.
    """

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as handle:

        for group in groups:

            representative = group[0]

            sequence = representative[
                sequence_key
            ]

            transcripts = ",".join(
                item["transcript_id"]
                for item in group
            )

            proteins = ",".join(
                item["protein_id"]
                for item in group
            )

            header = (
                f">{gene}|"
                f"{representative[isoform_key]}|"
                f"transcripts={transcripts}|"
                f"proteins={proteins}|"
                f"length={len(sequence)}"
            )

            if variant:

                header += (
                    f"|variant={variant}"
                )

            handle.write(
                header + "\n"
            )

            handle.write(
                wrap_sequence(
                    sequence
                )
                + "\n"
            )


# ============================================================
# RUN ANALYSIS
# ============================================================

analysis = analyse_all_isoforms(
    gene_symbol=gene_name,
    chromosome=variant_chromosome,
    position=variant_position,
    reference=variant_reference_bases,
    alternate=variant_alternate_bases,
)

summary_df = make_summary_dataframe(
    analysis["results"]
)

print_results(
    analysis,
    summary_df,
)


# ============================================================
# SAVE CSV AND FASTA
# ============================================================

if SAVE_FILES:

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    summary_csv = (
        OUTPUT_DIR
        / (
            f"{gene_name}_"
            f"all_transcripts_and_isoforms.csv"
        )
    )

    reference_fasta = (
        OUTPUT_DIR
        / (
            f"{gene_name}_"
            f"unique_reference_isoforms.fasta"
        )
    )

    alternate_fasta = (
        OUTPUT_DIR
        / (
            f"{gene_name}_"
            f"variant_resolved_"
            f"alternate_isoforms.fasta"
        )
    )

    summary_df.to_csv(
        summary_csv,
        index=False,
    )

    write_fasta(
        groups=analysis[
            "reference_groups"
        ],
        output_path=
            reference_fasta,
        gene=gene_name,
        sequence_key=
            "reference_protein",
        isoform_key=
            "unique_reference_isoform",
    )

    write_fasta(
        groups=analysis[
            "alternate_groups"
        ],
        output_path=
            alternate_fasta,
        gene=gene_name,
        sequence_key=
            "alternate_protein",
        isoform_key=
            "unique_alternate_isoform",
        variant=analysis[
            "variant"
        ],
    )

    print("\nSaved files:")

    print(
        summary_csv.resolve()
    )

    print(
        reference_fasta.resolve()
    )

    print(
        alternate_fasta.resolve()
    )


# ============================================================
# READY FOR ESM
# ============================================================

# One entry for every UNIQUE reference protein sequence.
reference_isoforms_for_esm = {
    group[0][
        "unique_reference_isoform"
    ]:
    group[0][
        "reference_protein"
    ]
    for group in analysis[
        "reference_groups"
    ]
}


# Only alternate proteins that could be directly reconstructed.
alternate_isoforms_for_esm = {
    group[0][
        "unique_alternate_isoform"
    ]:
    group[0][
        "alternate_protein"
    ]
    for group in analysis[
        "alternate_groups"
    ]
}


print(
    "\nReference isoforms ready for ESM:",
    list(
        reference_isoforms_for_esm
    ),
)

print(
    "Alternate isoforms ready for ESM:",
    list(
        alternate_isoforms_for_esm
    ),
)


GENE AND VARIANT
Gene: APOL4
Ensembl gene ID: ENSG00000100336
Gene location: chr22:36189124-36204840
Gene strand: -
Variant: chr22:36201698:A>C
Variant inside gene span: True
Most severe VEP consequence: splice_donor_variant
Translated Ensembl transcripts: 11
Unique reference protein isoforms: 7
Variant-resolved alternate protein isoforms: 0

TRANSCRIPT-TO-ISOFORM SUMMARY


,unique_reference_isoform,transcript_id,protein_id,reference_aa_length,is_ensembl_canonical,mane_select,appris,ccds,consequence_terms,direct_cds_overlap,amino_acid_change,hgvsp,protein_changed,alternate_status
0,ISOFORM_1,ENST00000683024,ENSP00000507418,348,True,True,A2,CCDS74851.1,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
1,ISOFORM_1,ENST00000332987,ENSP00000333229,348,False,False,A2,CCDS74851.1,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
2,ISOFORM_1,ENST00000457630,ENSP00000409085,348,False,False,A2,CCDS74851.1,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
3,ISOFORM_1,ENST00000616056,ENSP00000483497,348,False,False,A2,CCDS74851.1,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
4,ISOFORM_1,ENST00000684666,ENSP00000506782,348,False,False,A2,CCDS74851.1,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
5,ISOFORM_2,ENST00000352371,ENSP00000338260,351,False,False,P2,CCDS74852.1,intron_variant,False,None,None,None,not_resolved_no_direct_cds_change
6,ISOFORM_3,ENST00000880614,ENSP00000550673,290,False,False,None,None,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
7,ISOFORM_4,ENST00000397275,ENSP00000380445,107,False,False,None,CCDS93158.1,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change
8,ISOFORM_5,ENST00000493203,ENSP00000432133,70,False,False,None,None,"splice_donor_variant,NMD_transcript_variant",False,None,None,None,not_resolved_no_direct_cds_change
9,ISOFORM_6,ENST00000419360,ENSP00000395548,64,False,False,None,None,splice_donor_variant,False,None,None,None,not_resolved_no_direct_cds_change



UNIQUE REFERENCE PROTEIN ISOFORMS

>APOL4|ISOFORM_1
Transcripts: ENST00000683024, ENST00000684666, ENST00000457630, ENST00000616056, ENST00000332987
Protein IDs: ENSP00000507418, ENSP00000506782, ENSP00000409085, ENSP00000483497, ENSP00000333229
Length: 348 aa
Ensembl canonical: True
MANE Select: True
MGSWVQLITSVGVQQNHPGWTVAGQFQEKKRFTEEVIEYFQKKVSPVHLKILLTSDEAWKRFVRVAELPREEADALYEAL
KNLTPYVAIEDKDMQQKEQQFREWFLKEFPQIRWKIQESIERLRVIANEIEKVHRGCVIANVVSGSTGILSVIGVMLAPF
TAGLSLSITAAGVGLGIASATAGIASSIVENTYTRSAELTASRLTATSTDQLEALRDILRDITPNVLSFALDFDEATKMI
ANDVHTLRRSKATVGRPLIAWRYVPINVVETLRTRGAPTRIVRKVARNLGKATSGVLVVLDVVNLVQDSLDLHKGAKSES
AESLRQWAQELEENLNELTHIHQSLKAG

>APOL4|ISOFORM_2
Transcripts: ENST00000352371
Protein IDs: ENSP00000338260
Length: 351 aa
Ensembl canonical: False
MANE Select: False
MEGAALLKIFVVCIWVQQNHPGWTVAGQFQEKKRFTEEVIEYFQKKVSPVHLKILLTSDEAWKRFVRVAELPREEADALY
EALKNLTPYVAIEDKDMQQKEQQFREWFLKEFPQIRWKIQESIERLRVIANEIEKVHRGCVIANVVSGSTGILSVIGVML
APFTAGLSLSITAAGVGLGIASATAGIASSIVENTYTRSAELTASR